## 1. Importar librerías

Framework: Pytorch

In [ ]:
!pip install opencv-python
!pip install -q torchinfo
#!pip install -q torch torchvision torchinfo tensorboard scikit-learn seaborn matplotlib numpy pandas opencv-python pillow h5py scipy tqdm

In [ ]:
# Librerías estándar de Python
import os
import json
import gc
import time
import random
import warnings
from glob import glob
from pathlib import Path
from math import sqrt
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple, Union
from dataclasses import dataclass

# Librerías de manejo de datos y matemáticas
import numpy as np
import pandas as pd
import h5py

# Procesamiento de imágenes
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter

# Visualización y barras de progreso
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, random_split
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

# Deep Learning: Torchvision (Computer Vision)
import torchvision
from torchvision import datasets
from torchvision import models
import torchvision.transforms as T
from torchvision.transforms import v2 as transforms
from torchvision.ops import Conv2dNormActivation

%matplotlib inline
warnings.filterwarnings("ignore")

## 2. Descarga del dataset

Vamos a cargar las imágenes, que se encuentran divididas en varias carpetas. Necesitamos definir la estructura de directorios donde se alojarán los datos. Cada carpeta contiene 1000 imágens, excepto la última que contiene 1109 para completar las 5109 totales del dataset. El 70% de las imágenes están etiquetadas, las utilizaremos como conjunto para training y validation. El 30% de imágenes restantes serán empleadas para el conjunto de test en modo 'blind'.

| datos | nº imágenes | % |
| --- | --- | --- |
| training | 3609 | 70,64% |
| test | 1500 | 29,36% |

In [ ]:
%%bash
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/EWf9gbkfUH5Jg7ixk9l763ABQ2smxVMK59QXzysE3Rgllg?e=bMwv1h&download=1" -O "nwpu_crowd_json.zip"
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/ETyX9hxWI8RFlL09pdeAK3oBChI0v4rneiLi3bZ4_0pt4w?e=qsnuCy&download=1" -O "nwpu_crowd_images_1.zip"
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/ESKJnDj6QL9Ki7AH-DhBH5QBvR3VvsA05Yw-RqTHAx89VQ?e=kXcotR&download=1" -O "nwpu_crowd_images_2.zip"
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/Ea0BkEVwEyNDh75bZ3NXPukB4CZuo5Efh0btrYdQGbYE4g?e=tDxXIh&download=1" -O "nwpu_crowd_images_3.zip"
wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/EX3GjMQqlZROmVNeqGSnYh4B66K0VobqBc2EznW7xuFTWQ?e=3hgZLA&download=1" -O "nwpu_crowd_images_4.zip"

Estructura de directorios del entorno

In [ ]:
%%bash
mkdir -p data/json
mkdir -p data/images

unzip -q "nwpu_crowd_json.zip" -d data/json
unzip -q "nwpu_crowd_images_1.zip" -d data/images
unzip -q "nwpu_crowd_images_2.zip" -d data/images
unzip -q "nwpu_crowd_images_3.zip" -d data/images
unzip -q "nwpu_crowd_images_4.zip" -d data/images

In [ ]:
%%bash

mkdir -p data/images/training

for i in $(seq -f "%04g" 1 3609); do
  mv "data/images/${i}.jpg" "data/images/training/" 2>/dev/null || true
done

echo "Training:" $(ls data/images/training | wc -l) "imágenes"

Training: 3609 imágenes


Comprobaciones iniciales del dataset

In [ ]:
train_root = os.path.join("data", "images", "training")

#Comprobamos dimensiones de alguna de las imágenes (p.ej. 10 primeras)
for i, file in enumerate(sorted(os.listdir(train_root))[:10]):
    path = os.path.join(train_root, file)
    with Image.open(path) as img:
        print(f"{file}: {img.size}  (ancho x alto)")


0001.jpg: (3436, 2376)  (ancho x alto)
0002.jpg: (1920, 1280)  (ancho x alto)
0003.jpg: (2362, 1257)  (ancho x alto)
0004.jpg: (4884, 3072)  (ancho x alto)
0005.jpg: (4032, 3024)  (ancho x alto)
0006.jpg: (1600, 1066)  (ancho x alto)
0007.jpg: (1584, 1153)  (ancho x alto)
0008.jpg: (3872, 2592)  (ancho x alto)
0009.jpg: (4964, 3456)  (ancho x alto)
0010.jpg: (4691, 3126)  (ancho x alto)


Cálculo de la media y desviación de las imágenes

In [ ]:
means, stds = [], []

# !!!!!!! YA SE HA CALCULADO !!!!!!!
## NO HACE FALTA EJECUTAR SIEMPRE ##

for filename in tqdm(os.listdir(train_root)):
    img = cv2.imread(os.path.join(train_root, filename))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    means.append(np.mean(img, axis=(0,1))) #media por canal (R, G, B), promedio sobre las dimensiones alto y ancho
    stds.append(np.std(img, axis=(0,1)))

mean = np.mean(means, axis=0)
std = np.mean(stds, axis=0)

print("Mean:", mean)
print("Std:", std)

100%|██████████| 3109/3109 [40:53<00:00,  1.27it/s]

Mean: [0.44722986 0.41087792 0.39603603]
Std: [0.26052581 0.24937533 0.2516997 ]


**Carga de anotaciones desde .json con el formato esperado**

In [ ]:
def points_format(p) -> Tuple[float, float]:
    """
    Convierte distintos formatos de punto en (x, y) float.
    Acepta:
      - [x, y] o (x, y)
      - {"x":..., "y":...} o {"X":..., "Y":...}
      - Otros dicts con valores numéricos (toma los dos primeros)
    """
    if isinstance(p, (list, tuple)) and len(p) >= 2:
        return float(p[0]), float(p[1])
    if isinstance(p, dict):
        for kx, ky in (("x","y"), ("X","Y"), ("cx","cy"), ("pos_x","pos_y")):
            if kx in p and ky in p:
                return float(p[kx]), float(p[ky])
        #En última instancia, coger los dos primeros valores
        vals = [v for v in p.values() if isinstance(v, (int, float))]
        if len(vals) >= 2:
            return float(vals[0]), float(vals[1])

    raise ValueError(f"Formato de punto no reconocido: {p}")

def load_points_from_json_dir(jsons_dir: str, ) -> Dict[str, List[List[float]]]:
    """
    Lee una carpeta con archivos JSON donde cada archivo tiene la estructura:
      {
        "img_id": "XXXX.jpg",
        "points": [[x,y], ...]
      }

    Devuelve:
      { "XXXX.jpg": [[x1,y1], [x2,y2], ...], ... }

    """

    jsons_path = Path(jsons_dir)
    if not jsons_path.exists() or not jsons_path.is_dir():
        raise FileNotFoundError(f"El directorio '{jsons_dir}' no existe o no es un directorio.")

    json_files = sorted([p for p in jsons_path.iterdir()
                         if p.is_file() and p.suffix.lower() == ".json"])

    annotations: Dict[str, List[List[float]]] = {}

    for jf in json_files:
        try:
            with jf.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            raise ValueError(f"Error al leer o cargar el JSON '{jf.name}'")

        if not isinstance(data, dict):
            raise ValueError(f"El archivo '{jf.name}' no contiene un diccionario JSON válido.")

        img_id = data.get("img_id")
        if not img_id:
            raise ValueError(f"El archivo '{jf.name}' no contiene 'img_id'.")

        raw_points = data.get("points", [])
        points: List[List[float]] = []

        for p in raw_points:
            try:
                x, y = points_format(p)
                points.append([x, y])
            except Exception:
                #Ignora puntos no válidos
                continue

        annotations[img_id] = points

    return annotations

In [ ]:
#Carga de las anotaciones a partir de los .json
ann = load_points_from_json_dir("data/json")

#Comprobación de las anotaciones
for idx, (img_id, points) in enumerate(ann.items()):
        if idx == 10:  #Ver solo los idx primeros
          break
        points_str = ",  ".join(f"({x},{y})" for x, y in points)
        print(f"[{img_id}] -> [{points_str}]")

## 3. Preprocesamiento

**Reproducibilidad experimental**

El objetivo de la función *set_seed* es garantizar resultados consistentes utilizando los mismos datos y parámetros, reducir la variabilidad no explicada entre distintas ejecuciones de entrenamiento. Si se cambia un parámetro (p.ej. learning rate) de un entrenamiento respecto de otro debemos ser capaces de explicar el efecto de este sobre los resultados.

In [ ]:
#Establecer semilla para que las ejecuciones sean reproducibles
def set_seed(seed):
    random.seed(seed)           #Módulos estándar de Python
    np.random.seed(seed)        #Operaciones de NumPy
    torch.manual_seed(seed)     #CPU y GPU por defecto

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)    #Todas las GPUs en el sistema

       #Modo de búsqueda donde cuDNN prueba múltiples algoritmos para una operación dada y selecciona el más rápido para el tamaño de entrada específico
       torch.backends.cudnn.benchmark = False #False = Evita la selección dinámica de algoritmos
       #Obliga a la biblioteca a utilizar únicamente algoritmos que son deterministas
       torch.backends.cudnn.deterministic = True

set_seed(28)

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:

    img_size: int = 512         #Tamaño de entrada (H=W)
    batch_size: int = 16        #Muestras de entrenamiento procesadas simultaneamente
    num_epochs: int = 50        #Número de iteraciones completas sobre el conjunto de datos

    learning_rate: float = 1e-5 #Tasa de aprendizaje
    weight_decay: float = 5e-4  #Coeficiente de regularización L2
    max_norm: float = 10.0

    ### Sigma / densidad (Desviación que controla el radio de dispersión espacial aplicado a cada anotación para generar el mapa de densidad)
    sigma_base: float = 20.0     #Sigma de referencia como punto de partida
    adaptive_sigma: bool = True  #Activar sigma adaptativo

    # Parámetros de sigma adaptativo (distancia media a vecinos)
    adaptive_k: int = 3          # k: nº de vecinos
    adaptive_beta: float = 0.3   # beta: factor multiplicador para la distancia media (controla la dispersión del kernel Gaussiano)

    # Límites razonables de sigma en la resolución de salida para evitar extremos numéricos
    min_sigma_out: float = 1.5
    max_sigma_out: float = 10.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu" #Hardware de cómputo

train_config = TrainingConfig()
device = train_config.device


**Utils para la generación del Ground Truth (mapas de densidad)**

In [ ]:
#Media y desviación previamente calculada
#mean = [0.44722986, 0.41087792, 0.39603603]
#std =  [0.26052581, 0.24937533, 0.2516997]

#Media y desviación propias de la normalización de VGG16
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]


def compute_adaptive_sigmas(points_xy: np.ndarray, k: int, beta: float) -> np.ndarray:
    """
    Calcula un sigma adaptativo por punto basado en distancia media a sus k vecinos más cercanos.

    points_xy: array shape (N, 2) en coordenadas (x, y).
    k: número de vecinos para estimar densidad local (típico 3).
    beta: factor multiplicador de la distancia media (control de dispersión).

    Devuelve:
        sigmas: array shape (N,) con sigma por punto (float).
    """
    #Si hay 0 o 1 punto, no se puede estimar vecindad: devolvemos sigma=1 por defecto
    if points_xy.shape[0] <= 1:
        return np.ones((points_xy.shape[0],), dtype=np.float32)

    #Cálculo de la matriz de distancias euclídeas NxN (coste O(N^2))
    diffs = points_xy[:, None, :] - points_xy[None, :, :]               # (N, N, 2)
    dists = np.sqrt(np.sum(diffs ** 2, axis=-1))                        # (N, N)

    #diagonal a infinito para que el punto no se considere a sí mismo como vecino
    np.fill_diagonal(dists, np.inf)

    #Se toman las k distancias más pequeñas por punto
    k_eff = min(k, points_xy.shape[0] - 1)                              # k efectivo (no puede exceder N-1)
    knn = np.partition(dists, kth=k_eff - 1, axis=1)[:, :k_eff]         # (N, k_eff)

    #Distancia media a k vecinos
    mean_knn = np.mean(knn, axis=1)                                     # (N,)

    #Sigma adaptativo: beta * distancia_media
    sigmas = beta * mean_knn

    #Si por algún caso extremo sigma sale 0 (puntos duplicados), se fuerza a mínimo > 0
    sigmas = np.maximum(sigmas, 1e-3).astype(np.float32)

    return sigmas


def create_density_map_adaptive(
    points: list,
    H: int,
    W: int,
    adaptive: bool,
    sigma_base: float,
    k: int,
    beta: float,
    min_sigma: float,
    max_sigma: float
) -> np.ndarray:
    """
    Crea un mapa de densidad (H, W) a partir de puntos
    - Si adaptive=False: usa sigma_base global
    - Si adaptive=True: usa sigma por punto calculado con kNN

    - Se usa un kernel Gaussiano por punto y se suma sobre el mapa
    - Se recorta ROI (Region of Interest) alrededor del punto para evitar coste O(H*W*N).
    """
    density = np.zeros((H, W), dtype=np.float32)

    #Si no hay puntos, devolvemos mapa vacío
    if points is None or len(points) == 0:
        return density

    pts = np.asarray(points, dtype=np.float32)  # shape (N,2) en (x,y)

    #Calculamos sigma por punto si es adaptativo, si no, todos igual
    if adaptive:
        sigmas = compute_adaptive_sigmas(pts, k=k, beta=beta)  # (N,)
    else:
        sigmas = np.full((pts.shape[0],), float(sigma_base), dtype=np.float32)

    #Límites para evitar kernels demasiado pequeños o grandes en el output
    sigmas = np.clip(sigmas, min_sigma, max_sigma)

    #Generamos un kernel Gaussiano por punto y lo acumulamos en una ventana local o región de interés (ROI)
    for (x, y), s in zip(pts, sigmas):
        #Redondeo al píxel más cercano
        xi = int(round(float(x)))
        yi = int(round(float(y)))

        #Ignorar puntos fuera de la imagen
        if xi < 0 or xi >= W or yi < 0 or yi >= H:
            continue

        #Radio del ROI = 3*sigma (más allá de 3 desviaciones estándar (3σ) la función Gaussiana ~= 0)
        radius = max(1, int(round(3.0 * float(s))))

        #Coordenadas del ROI recortadas a los bordes del mapa
        x1 = max(0, xi - radius)
        x2 = min(W, xi + radius + 1)
        y1 = max(0, yi - radius)
        y2 = min(H, yi + radius + 1)

        #Tamaño del kernel para el ROI actual
        kW = x2 - x1
        kH = y2 - y1

        #Parche/matriz del tamaño del ROI
        patch = np.zeros((kH, kW), dtype=np.float32)
        patch[yi - y1, xi - x1] = 1.0

        #Al patch se le aplica desenfoque gaussiano, OpenCV GaussianBlur
        patch = cv2.GaussianBlur(patch, ksize=(0, 0), sigmaX=float(s), sigmaY=float(s)) # ksize=(0,0) deja que OpenCV lo derive del sigma

        #Sumamos el patch al mapa de densidad global
        density[y1:y2, x1:x2] += patch

    return density


class CSRNet_FromGroundTruths(Dataset):
    """
    Clase tipo Dataset que define el acceso a cada elemento del conjunto de datos.

    images_dir: directorio con las imágenes
    annotations: diccionario de las coordenadas de los puntos de cada imagen (img_name: [[x1,y1], [x2,y2], ...])
    sigma: desviación estándar del kernel gaussiano
    image_size:  tamaño (height, width) al que las imagenes son reescaladas
    mean/std: valores de media y desiación para la normalización de las imágenes

    Devuelve:
      - img_tensor: imagen tensor (3, H, W)
      - gt_tensor: Ground Truth (mapa de densidad) (1, H, W)
      - pts: número de puntos (int)
    """

    def __init__(self, images_dir: str, annotations: dict = None):
        super().__init__()

        self.image_size = train_config.img_size
        self.sigma = train_config.sigma_base
        self.mean = mean
        self.std = std

        #Tranformaciones que se van a aplicar a la imagen
        self.transform = T.Compose([
            T.Resize((self.image_size, self.image_size)),
            T.ToTensor(),
            T.Normalize(mean=self.mean, std=self.std)
        ])

        # path de las imágenes
        images_glob = os.path.join(images_dir, "*.jpg")
        self.image_paths = sorted(glob(images_glob))
        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {images_dir} with pattern '*.jpg'")

        #Normalización de las claves de annotations para que coincidan con los nombres de las imágenes que se van a cargar
        # "data/train/0001.jpg" -> "0001.jpg"
        self.annotations = None
        if annotations is not None:
            new_ann = {}
            for k, v in annotations.items():
                new_ann[Path(k).name] = v
            self.annotations = new_ann

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        #Cargar la imagen original y recoger las dimensiones
        img_path = self.image_paths[index]
        img = Image.open(img_path).convert("RGB")
        orig_W, orig_H = img.size

        #Aplicar las transformaciones
        img_t = self.transform(img)

        img_name = Path(img_path).name
        if self.annotations is not None:
            pts_list = self.annotations.get(img_name, [])
            pts = len(pts_list) #Número de puntos en la imagen

            #En CSRNet la salida del modelo suele ser 1/8 del tamaño de la imagen
            #(evita problemas de downsampling, que ocurrirían si primero se hiciera la densidad al tamaño completo y luego la redujese a 1/8)
            out_H = self.image_size // 8
            out_W = self.image_size // 8

            #Escalar las coordenadas originales al nuevo tamaño
            x_out_scaled = out_W / float(orig_W)
            y_out_scaled = out_H / float(orig_H)
            scaled_pts_out = [(x * x_out_scaled, y * y_out_scaled) for (x, y) in pts_list]

            #Si sigma adaptativo = False:
            # se usasigma_base reescalado a la resolución de salida
            #Si sigma adaptativo = True:
            # se calcula sigma por punto a partir de distancias entre puntos en la resolución de salida
            avg_scale_out = 0.5 * (x_out_scaled + y_out_scaled)  # escala media

            sigma_base_out = float(train_config.sigma_base) * float(avg_scale_out)  # sigma de referencia en salida

            gt_resized = create_density_map_adaptive(
                points=scaled_pts_out,                       # puntos escalados a (out_W, out_H)
                H=out_H,
                W=out_W,
                adaptive=train_config.adaptive_sigma,
                sigma_base=sigma_base_out,
                k=train_config.adaptive_k,                   # nº vecinos para estimar densidad local
                beta=train_config.adaptive_beta,             # factor multiplicador de distancia media
                min_sigma=train_config.min_sigma_out,        # límite inferior
                max_sigma=train_config.max_sigma_out         # límite superior
            ).astype(np.float32)

        else:
            #Si no hay anotaciones, todo ceros en el mapa de densidad
            pts = 0
            out_H = max(1, self.image_size // 8)
            out_W = max(1, self.image_size // 8)
            gt_resized = np.zeros((out_H, out_W), dtype=np.float32)

        #La matriz del mapa de densidad se convierte a tensor (1, out_H, out_W) (En mapas de densidad un único canal de color)
        gt_tensor = T.ToTensor()(gt_resized.astype(np.float32))

        return img_t, gt_tensor, pts

In [ ]:
def check_dataset(dataset, n=5):
    """
    Función para comprobar que el dataset está bien cargado + prueba de tolerancia (no usado de momento)
    """
    tol = 5
    N = min(len(dataset), n)
    print(f"Tamaño del dataset: {len(dataset)}. Primeras {N} imágenes...\n")
    for i in range(N):
        img, gt, pts = dataset[i]
        #Chequeos básicos
        img_ok = (isinstance(img, torch.Tensor) and img.dtype.is_floating_point and img.ndim == 3 and img.shape[0] == 3)
        gt_ok = (isinstance(gt, torch.Tensor) and gt.ndim == 3 and gt.shape[0] == 1)

        gt_sum = float(gt.sum().item())
        pts_int = int(pts)
        match = abs(gt_sum - pts_int) <= tol

        print(f"[{i}] img: shape={tuple(img.shape)}, dtype={img.dtype}, ok={img_ok}")
        print(f"      gt: shape={tuple(gt.shape)}, dtype={gt.dtype}, sum={gt_sum:.3f}, pts={pts_int}, match={match}")
        if not gt_ok:
            print("     -> [WARNING] gt tensor shape/dtype erróneo.")
        if not img_ok:
            print("     -> [WARNING] image tensor shape/dtype erróneo.")
        if not match:
            print("     -> [WARNING] gt.sum() difiere de pts por más de la tolerancia")
        print()


In [ ]:
dataset_checking = CSRNet_FromGroundTruths(images_dir=train_root, annotations=ann)
check_dataset(dataset_checking, n=10)

Tamaño del dataset: 3609. Primeras 10 imágenes...

[0] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=45.103, pts=45, match=True

[1] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=0.000, pts=0, match=True

[2] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=0.000, pts=0, match=True

[3] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=111.572, pts=108, match=True

[4] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=38.064, pts=37, match=True

[5] img: shape=(3, 512, 512), dtype=torch.float32, ok=True
      gt: shape=(1, 64, 64), dtype=torch.float32, sum=861.982, pts=839, match=False
     -> [WARNING] gt.sum() difiere de pts por más de la tolerancia

[6] img: shape=(3, 512, 512), dtype

## 4. Construcción del modelo

In [ ]:
def make_layers(config_architecture, in_channels=3, batch_norm=False, dilation=False):

    """
    Función para crear las capas de la red neuronal.
    - config_architecture: configuración de la arquitectura de capas a construir
    - in_channels: número de canales de entrada (3 para imágenes RGB)
    - batch_norm: si poner o no Batch Normalization
    - dilation: usar o no dilated convolutions

    """
    #Si dilation=True entonces el kernel de convolución se expande para capturar contexto más grande sin perder resolución espacial
    d_rate = 2 if dilation else 1
    layers = []
    for out_channels in config_architecture:
        if out_channels == 'M': #Añade capa de MaxPooling que reduce por dos
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:        #Si es un número, capa de Convolución
            conv2d = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               padding=d_rate, dilation=d_rate)
            if batch_norm:
                #Normalización si se indica, se coloca después de la convolución y antes de la activación (ReLU) para estabilidad del entrenamiento
                layers += [conv2d, nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = out_channels #Para la próxima convolución, los canales de entrada serán los canales de salida de la capa actual
    return nn.Sequential(*layers) #Creación del bloque secuencial de capas

class CSRNet(nn.Module):

    """
    Clase del modelo CSRNet.
    """
    def __init__(self, load_weights=False):
        super(CSRNet, self).__init__()

        #Bloque de capas frontend típico de VGG
        self.frontend_feature = [64, 64, 'M',
                                128, 128, 'M',
                                256, 256, 256, 'M',
                                512, 512, 512]

        #Dilated backend standard de CSRNet
        self.backend_feature = [512, 512, 512, 256]

        self.frontend = make_layers(self.frontend_feature)
        self.backend = make_layers(self.backend_feature, in_channels=512, dilation=True)

        #Bloque de capas output típico de CSRNet
        self.output_layer = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            #nn.ReLU(inplace=True)   #evita negativos en VERSION CSRNet Estable, sin ella versión CSRNet Pura
        )

        #Carga de los pesos preentrenados del modelo VGG16
        if not load_weights:
            vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
            vgg_conv = [m for m in vgg.features if isinstance(m, nn.Conv2d)]
            self_conv = [m for m in self.frontend if isinstance(m, nn.Conv2d)]
            for s, v in zip(self_conv, vgg_conv):
                s.weight.data[:] = v.weight.data.clone()
                s.bias.data[:] = v.bias.data.clone()

        #Inicialización de las capas del backend y output (frontend se mantienen los preentrenados)
        self.initialize_weights(scope='backend_output')

        #Inicialización de la última capa (std=0.001 y bias=0)
        #Para que la red empiece prediciendo un mapa de densidad cercano a cero y evitar Loss inicial enorme e inestabilidad
        last = self.output_layer[-1]
        if isinstance(last, nn.Conv2d):
            nn.init.normal_(last.weight, std=1e-3)
            if last.bias is not None:
                nn.init.constant_(last.bias, 0.0)

    #Función de propagación hacia adelante en la red neuronal
    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


    #Función de inicialización de pesos
    def initialize_weights(self, scope='all'):
        modules = []
        if scope == 'all':
            modules = self.modules()
        elif scope == 'backend_output':
            modules = list(self.backend.modules()) + list(self.output_layer.modules())
        for m in modules:
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


## 5. Funciones de recogida de Logs

In [ ]:
class LogsFunction:

    def __init__(self, output_dir: str, num_epochs: int):
        self.output_dir = output_dir
        self.num_epochs = num_epochs

        log_files_count = len(glob(os.path.join(self.output_dir, "Log_*.txt")))

        self.output_file_dir = os.path.join(
            self.output_dir,
            f"Log_{log_files_count}_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.txt"
        )

        self.create_log_file()

    def create_log_file(self):
        with open(self.output_file_dir, 'w', encoding="utf-8") as wr:
            wr.write(f"Fecha de creación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    def log_hyperparams(self, **config):
        #Bbloque de hiperparámetros para la trazabilidad
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            wr.write("\n======= Configuracion del entrenamiento =======\n")
            for key, value in config.items():
                wr.write(f"- {key}: {value}\n")
            wr.write("===============================================\n\n")

    def log_results(self, epoch: int, **metrics):
        with open(self.output_file_dir, 'a', encoding="utf-8") as writer:
            log = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [{epoch}/{self.num_epochs}] "
            for key, value in metrics.items():
                try:
                    value = float(value)
                    log += f"| {key}: {value:.6f} "
                except Exception:
                    log += f"| {key}: {value} "
            writer.write(log + "\n")

    def write(self, msg: str):
        #Mensaje general
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] {msg}\n")
        print(msg)

    def log_error(self, err: str):
        #Mensaje de error/warnings
        with open(self.output_file_dir, 'a', encoding="utf-8") as wr:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            wr.write(f"[{timestamp}] {err}\n")
        print(err)

## 6. Entrenamiento del modelo

In [ ]:
def count_metrics(p_gt, gt):

    #Paso de densidad a número de personas, p_gt y gt: (B,1,H,W)
    pred_counts = p_gt.view(p_gt.size(0), -1).sum(dim=1)  # p_gt.view(p_gt.size(0), -1): paso de (B,1,H,W) a (B, H*W), pred_counts tiene un valor por imagen
    gt_counts   = gt.view(gt.size(0), -1).sum(dim=1)

    #error absoluto y cuadrático de cada imagen
    abs_errs = torch.abs(pred_counts - gt_counts) # (B,)
    sq_errs  = (pred_counts - gt_counts)**2

    #Devuelve la suma de los errores del batch
    return abs_errs.sum().item(), sq_errs.sum().item()

def train_batch(model, optimizer, data, criterion):
    #modelo en modo entrenamiento
    model.train()

    img, gt, pt = data
    img = img.to(device)
    gt  = gt.to(device)

    if isinstance(pt, torch.Tensor):
        pt = pt.to(device)

    #Limpieza de gradientes (más eficiente en memoria que poner todos los gradientes a 0)
    optimizer.zero_grad(set_to_none=True)

    #Forward pass para obtener las predicciones de la imagen
    preds = model(img)   #(B,1,Hout,Wout)

    """
    # Debug: predicciones y GT antes de backprop
    with torch.no_grad():
        B = preds.size(0)
        pred_counts = preds.view(B, -1).sum(dim=1).cpu().numpy()
        gt_counts   = gt.view(B, -1).sum(dim=1).cpu().numpy()
        print("batch gt counts (first 8):", gt_counts[:8])
        print("batch pred counts (first 8):", pred_counts[:8])
    """
    #Cálculo del loss y backpropagation
    loss = criterion(preds, gt)
    loss.backward()

    #clip gradients para evitar gradientes explosivos
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.max_norm)

    #actualizar todos los pesos del modelo con los gradientes
    optimizer.step()

    """
    #Añadido para debug: pesos en la última capa
    grad_norm = 0.0
    for p in model.output_layer.parameters():
        if p.grad is not None: grad_norm += p.grad.data.norm(2).item()**2
    grad_norm = grad_norm**0.5
    print("last-layer grad norm:", grad_norm)

    with torch.no_grad():
        try:
            last_conv = model.output_layer[-1]
            if isinstance(last_conv, torch.nn.Conv2d):
                w = last_conv.weight.detach().cpu()
                b = last_conv.bias.detach().cpu() if last_conv.bias is not None else None
                print("after step - last conv - weight mean,std:", w.mean().item(), w.std().item(),
                      " bias mean,std:", None if b is None else (b.mean().item(), b.std().item()))
        except Exception as e:
            print("Error: no se puedo leer last_conv ", e)
    """

    #Métricas de conteo
    abs_err, sq_err = count_metrics(preds, gt)

    #pt_loss como MAE medio del batch
    pt_loss = abs_err / float(preds.size(0))

    #mover escalares fuera de los tensores
    loss_item = loss.item()
    #pt_loss_item = pt_loss.item()
    pt_loss_item = pt_loss

    #liberar memoria
    del loss, pt_loss
    del img, gt, preds

    return loss_item, pt_loss_item, abs_err, sq_err

def validate_batch(model, data, criterion):
    """
    Evalúa un batch de imágenes
    Devuelve:
      - loss_item: loss medio del batch
      - pt_loss_item: MAE medio por imagen del conteo
      - abs_err: suma de errores absolutos de conteo en el batch
      - sq_err: suma de errores cuadráticos de conteo en el batch
    """
    #Modelo en modo evaluación
    model.eval()

    img, gt, pt = data
    img = img.to(device)
    gt  = gt.to(device)
    if isinstance(pt, torch.Tensor):
        pt = pt.to(device)

    #No se hace torch.no_grad() porque se hace en la función principal antes de llamar a esta
    p_gt = model(img)

    loss = criterion(p_gt, gt)                  #Loss
    abs_err, sq_err = count_metrics(p_gt, gt)   #Métricas de conteo

    # pt_loss como MAE medio por imagen del batch
    pt_loss = abs_err / float(p_gt.size(0))

    #Convertir a escalares python antes de liberar tensores
    loss_item = float(loss.item())
    pt_loss_item = float(pt_loss)

    #Liberación de tensores de memoria (si no se acaba la RAM de la GPU en Google Colab)
    del loss, pt_loss, p_gt
    del img, gt, pt

    return loss_item, pt_loss_item, abs_err, sq_err


def visualize_density_maps(model, data, save_path=None):
    """
    Guarda imagen original, GT (mapa de densidad original) y predicción (mapa de desidad) en una imagen PNG
    """
    model.eval()
    img, gt, _ = data
    img = img.to(device)

    with torch.no_grad():
        pred = model(img)

    img_np = img[0].cpu().numpy().transpose(1,2,0)    # mueve el tensor a CPU para poder convertirlo en array NumPy, y ambia el orden de canales (C, H, W) → (H, W, C)
    img_np = img_np * np.array(std) + np.array(mean)  #desnormalizar la imagen
    img_np = np.clip(img_np, 0, 1)                    #Garantiza que los píxeles estén entre 0 y 1, para evitar: colores saturados, valores negativos

    gt_np = gt[0].squeeze().cpu().numpy()             # añadido .squeeze() para eliminar dimensiones de tamaño 1, (1, 1, H, W) → (H, W), convertir a Numpy para poder visualizarlos
    pred_np = pred[0].squeeze().cpu().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    axs[0].imshow(img_np)
    axs[0].set_title("Imagen")

    axs[1].imshow(gt_np, cmap='jet')
    axs[1].set_title(f"GT Density (count={gt_np.sum():.1f})")

    axs[2].imshow(pred_np, cmap='jet')
    axs[2].set_title(f"Prediction Density (count={pred_np.sum():.1f})")

    for a in axs:
        a.axis("off")

    if save_path:
        plt.savefig(save_path)
    plt.close()

**Función principal de entrenamiento**

In [ ]:
VIZ = 5

def CrowdCounting(train_root : str, output_dir : str, total_epochs : int, model_pth : Union[str, None] = None):

    #Directorios para los resultados de salida
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "saved_weights"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)

    writer = SummaryWriter(output_dir) #TensorBoard logger

    #Logs
    logger = LogsFunction(output_dir=output_dir, num_epochs=total_epochs)
    logger.log_hyperparams(
      batch_size=train_config.batch_size,
      num_epochs=train_config.num_epochs,
      learning_rate=train_config.learning_rate,
      max_norm=train_config.max_norm,
      sigma_base=train_config.sigma_base,
      adaptive_sigma=train_config.adaptive_sigma,
      adaptive_k=train_config.adaptive_k,
      adaptive_beta=train_config.adaptive_beta,
      max_sigma_out=train_config.max_sigma_out,
      min_sigma_out=train_config.min_sigma_out,
      img_size=train_config.img_size,
      optimizer="Adam",
      scheduler="MultiStepLR(milestones=[25, 40], gamma=0.3)"
    )

    #Carga del modelo
    model = CSRNet().to(device)

    #Logs
    if model_pth:
        logger.write("[INFO] Se han proporcionado pesos del modelo para cargarse.")
        try:
            model.load_state_dict(torch.load(model_pth), strict=True)
            logger.log_error("[INFO] Pesos cargados correctamente con strict=True.")
        except RuntimeError as e:
            logger.log_error(f"[WARNING] Runtime error mientras se cargaban los pesos del modelo: {e}")
        except FileNotFoundError as e:
            logger.log_error(f"[ERROR] Archivo no encontrado: {e}")
        except Exception as e:
            logger.log_error(f"[ERROR] Ocurrió un error al cargar los pesos del modelo: {e}")
    else:
        logger.write("[INFO] No se han proporcionado pesos iniciales para el modelo. Entrenando desde cero (from scratch).")

    #Criterio
    #criterion = nn.L1Loss()
    criterion = nn.MSELoss()

    optimizer = optim.Adam(model.parameters(), lr=train_config.learning_rate, weight_decay=train_config.weight_decay)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
      optimizer, milestones=[25, 40], gamma=0.3
    )

    #Obtener los elementos del dataset, cada elemento está formado por: (img_t, gt_tensor, pts)
    dataset = CSRNet_FromGroundTruths(images_dir=train_root, annotations=ann)

    #Pproporciones de train/val
    train_size  = int(0.8  *  len(dataset))     #80%  train
    val_size =  len(dataset)  - train_size      #20%  val

    #Dividir aleatoriamente
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    trn_dl = DataLoader(train_dataset, batch_size=train_config.batch_size, shuffle=True)
    val_dl = DataLoader(val_dataset, batch_size=train_config.batch_size,  shuffle=False)

    training_history = [] #Recogida de métricas

    best_loss = float('inf')
    best_val_mae = float("inf")

    for epoch in range(total_epochs):
        ### Marcar tiempo de inicio de época ###
        epoch_start = time.time()

        print(f"\nEpoch {epoch+1}/{total_epochs}")
        print("-" * 50)

        ############## Bucle Training ##############
        total_tr_loss = 0.0
        total_tr_pts_loss = 0.0
        train_abs_err = 0
        train_sq_err = 0

        for data in tqdm(trn_dl, desc="Training: "):
            loss, pts_loss, abs_err, sq_err = train_batch(model, optimizer, data, criterion)
            total_tr_loss+=loss
            total_tr_pts_loss+=pts_loss
            train_abs_err += abs_err
            train_sq_err += sq_err

        #Cálculo métricas Training
        avg_tr_loss = total_tr_loss / len(trn_dl)
        avg_tr_pt_loss = total_tr_pts_loss / len(trn_dl)

        train_mae = train_abs_err / len(train_dataset)   # dividir por número de imágenes, no de batches
        train_rmse = sqrt(train_sq_err / len(train_dataset))

        ############## Bucle Validation ##############
        total_val_loss = 0.0
        total_val_pts_loss = 0.0
        val_abs_err = 0
        val_sq_err = 0

        with torch.no_grad():
            for data in tqdm(val_dl, desc="Validating: "):
                val_loss, pts_loss, abs_err, sq_err = validate_batch(model, data, criterion)
                total_val_loss += val_loss
                total_val_pts_loss += pts_loss

                #Acumulación de errores para MAE/RMSE globales por imagen
                val_abs_err += abs_err
                val_sq_err += sq_err


        #Cálculo métricas Validation
        avg_val_loss = total_val_loss / len(val_dl)
        avg_val_pt_loss = total_val_pts_loss / len(val_dl)

        val_mae = val_abs_err / len(val_dataset)
        val_rmse = sqrt(val_sq_err / len(val_dataset))

        ### Calcular duración de la época ###
        epoch_duration = time.time() - epoch_start

        ### Obtener el Learning Rate actual ###
        current_lr = optimizer.param_groups[0]['lr']

        ################ LOGGING POR PANTALLA ###############
        print(f"Train Loss: {avg_tr_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Train Pt Loss: {avg_tr_pt_loss:.4f} | Val Pt Loss: {avg_val_pt_loss:.4f}")
        print(f"Train MAE:  {train_mae:.2f} | Val MAE:  {val_mae:.2f}")
        print(f"Train RMSE: {train_rmse:.2f} | Val RMSE: {val_rmse:.2f}")

        ############### TENSORBOARD LOGGING ############### ??????????
        writer.add_scalar("Loss/train", avg_tr_loss, epoch)
        writer.add_scalar("Loss/val", avg_val_loss, epoch)
        writer.add_scalar("MAE/train", train_mae, epoch)
        writer.add_scalar("MAE/val", val_mae, epoch)
        writer.add_scalar("RMSE/train", train_rmse, epoch)
        writer.add_scalar("RMSE/val", val_rmse, epoch)

        ### Guardar datos en la lista de historial ###
        training_history.append({
            'Epoch': epoch + 1,
            'LR': current_lr,
            'Time_Sec': round(epoch_duration, 2),
            'Train_Loss': avg_tr_loss,
            'Val_Loss': avg_val_loss,
            'Train_MAE': train_mae,
            'Val_MAE': val_mae,
            'Train_RMSE': train_rmse,
            'Val_RMSE': val_rmse
        })

        #Guardar el mejor modelo basándose en avg validation loss
        if avg_val_loss < best_loss:
            if not os.path.exists(os.path.join(output_dir, "saved_weights")):
                os.mkdir(os.path.join(output_dir, "saved_weights"))

            best_loss = avg_val_loss
            save_path = os.path.join(os.path.join(output_dir, "saved_weights"), f'Best_model_{epoch+1}.pth')
            torch.save(model.state_dict(), save_path)

        #Logs de métricas por época
        logger.log_results(epoch=epoch+1,
                           tr_loss = avg_tr_loss, val_loss = avg_val_loss,
                           tr_pt_loss = avg_tr_pt_loss, val_pt_loss = avg_val_pt_loss,
                           train_mae = train_mae, train_rmse = train_rmse,
                           val_mae = val_mae, val_rmse = val_rmse)

        #Scheduler step
        scheduler.step()

        ############### VISUALIZACIÓN de mapas de densidad (cada VIZ épocas) ################
        if epoch % VIZ == 0:
            sample = next(iter(val_dl))
            vis_path = os.path.join(output_dir, "visualizations", f"epoch_{epoch+1}.png")
            visualize_density_maps(model, sample, save_path=vis_path)
            print(f"[+] Visualización guardada en {vis_path}")

    writer.close()

    ### CREAR EXCEL DE MÉTRICAS DE ENTRENAMIENTO Y VALIDACIÓN ###
    print("[INFO] Generando Excel...")

    #Crear DataFrame
    df = pd.DataFrame(training_history)

    #Añadir columnas de configuración constantes
    df['Batch_Size'] = train_config.batch_size
    df['Image_Size'] = train_config.img_size
    df['Sigma'] = train_config.sigma_base
    df['Min_sigma_out'] = train_config.min_sigma_out
    df['Max_sigma_out'] = train_config.max_sigma_out
    df['Learning_Rate'] = train_config.learning_rate

    #Nombre del archivo Excel
    excel_path = os.path.join(output_dir, "metricas_training_validation.xlsx")

    #Guardar
    try:
        df.to_excel(excel_path, index=False)
        print(f"[ÉXITO] Métricas guardadas exitosamente en: {excel_path}")
    except Exception as e:
        print(f"[ERROR] No se pudo guardar el Excel: {e}")
        #Fallback a CSV si falla Excel
        csv_path = os.path.join(output_dir, "metricas_training_validation.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Guardado como CSV en su lugar: {csv_path}")

In [ ]:
output_dir = "output_crowd_counting" # Directorio de output
total_epoch = train_config.num_epochs
model_pth = None # Para modelo preentrenado

CrowdCounting(train_root, output_dir, total_epoch, model_pth)

[INFO] No se han proporcionado pesos iniciales para el modelo. Entrenando desde cero (from scratch).

Epoch 1/50
--------------------------------------------------


Validating: 100%|██████████| 46/46 [02:30<00:00,  3.28s/it]


Train Loss: 0.0782 | Val Loss: 0.0533
Train Pt Loss: 262.0773 | Val Pt Loss: 146.0308
Train MAE:  262.68 | Val MAE:  147.89
Train RMSE: 1010.14 | Val RMSE: 422.60
[+] Visualización guardada en output_crowd_counting/visualizations/epoch_1.png

Epoch 2/50
--------------------------------------------------


Validating: 100%|██████████| 46/46 [02:30<00:00,  3.26s/it]


Train Loss: 0.0605 | Val Loss: 0.0498
Train Pt Loss: 186.0158 | Val Pt Loss: 146.1778
Train MAE:  186.04 | Val MAE:  147.65
Train RMSE: 810.04 | Val RMSE: 430.84

Epoch 3/50
--------------------------------------------------


...

Validating: 100%|██████████| 46/46 [02:22<00:00,  3.10s/it]


Train Loss: 0.0454 | Val Loss: 0.0403
Train Pt Loss: 127.8964 | Val Pt Loss: 113.0135
Train MAE:  127.87 | Val MAE:  114.36
Train RMSE: 559.78 | Val RMSE: 310.93

Epoch 11/50
--------------------------------------------------


Validating: 100%|██████████| 46/46 [02:24<00:00,  3.14s/it]


Train Loss: 0.0433 | Val Loss: 0.0401
Train Pt Loss: 117.0952 | Val Pt Loss: 105.5199
Train MAE:  117.33 | Val MAE:  106.52
Train RMSE: 496.92 | Val RMSE: 317.32
[+] Visualización guardada en output_crowd_counting/visualizations/epoch_11.png

Epoch 12/50
--------------------------------------------------


Training:   5%|▍         | 9/181 [01:00<21:38,  7.55s/it]

In [ ]:
#Liberación de memoria
gc.collect()
torch.cuda.empty_cache()